In [10]:
import pandas as pd
import glob
import os
import json
import numpy as np

# Define directories
JSON_DIR = "../logs/facts_one_hop/nemotron-3-nano"
CSV_PATH = "../logs/facts_one_hop/nemotron-3-nano/analysis_by_gemini_3_flash.csv"
TRACES_PATH = "../logs/facts_one_hop/nemotron-3-nano/baseline_agent_traces_20260208_102329.json"

def load_no_search_baseline(json_dir, num_questions):
    pattern = os.path.join(json_dir, '*no_search*.json')
    files = glob.glob(pattern)
    
    if not files:
        print(f"Warning: No no-search JSON files found in {json_dir}")
        return None
        
    print(f"Found {len(files)} no-search run files.")
    
    correct_counts = np.zeros(num_questions, dtype=int)
    run_count = 0
    
    for filepath in files:
        try:
            with open(filepath, 'r') as f:
                data = json.load(f)
                data = sorted(data, key=lambda x: x['problem'])
                for i, item in enumerate(data):
                    if i < num_questions:
                        if item.get('sampler_correct', False):
                            correct_counts[i] += 1
                run_count += 1
        except Exception as e:
            print(f"Error loading {filepath}: {e}")
            
    if run_count == 0:
        return None
        
    is_stable_correct = (correct_counts == 5)
    print(f"Identified {is_stable_correct.sum()} questions with stable correct answers (5/5).")
    return is_stable_correct

def load_and_preprocess(csv_path, json_dir=None):
    if not os.path.exists(csv_path):
        print(f"Error: {csv_path} not found.")
        return None
    
    try:
        df = pd.read_csv(csv_path)
        df = df.sort_values('problem_id').reset_index(drop=True)
        print(f"Loaded raw data: {len(df)} rows.")
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return None

    column_mapping = {
        'judge1_confidence': 'pre_search_confidence',
        'judge2_hypothesis_correct': 'hypothesis_correct',
        'judge2_query_biased': 'is_search_query_biased',
        'judge2_snippet_has_answer': 'snippet_has_answer',
        'judge2_answer_flipped': 'answer_flipped'
    }
    df.rename(columns=column_mapping, inplace=True)

    bool_cols = ['baseline_correct', 'agent_correct', 'snippet_has_answer', 'answer_flipped', 'is_search_query_biased']
    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).map({'True': True, 'False': False, '1': True, '0': False, '1.0': True, '0.0': False})
            df[col] = df[col].fillna(False)

    if json_dir:
        stable_correct = load_no_search_baseline(json_dir, len(df))
        if stable_correct is not None:
             if len(stable_correct) == len(df):
                 df['baseline_correct'] = stable_correct
                 print("Updated 'baseline_correct' based on no-search JSON evaluations (5/5 correct).")
             else:
                 print(f"Warning: Length mismatch. DF: {len(df)}, No-Search: {len(stable_correct)}")

    if 'is_context_poisoning' not in df.columns:
        if 'baseline_correct' in df.columns and 'agent_correct' in df.columns:
            df['is_context_poisoning'] = df['baseline_correct'] & (~df['agent_correct'])

    if 'is_performative_ignorance' not in df.columns and 'pre_search_confidence' in df.columns:
        if 'baseline_correct' in df.columns:
            df['is_performative_ignorance'] = df['baseline_correct'] & df['pre_search_confidence'].isin(['TABULA_RASA', 'WEAK_GUESS'])

    if 'is_confirmation_bias' not in df.columns and 'pre_search_confidence' in df.columns:
        if 'is_search_query_biased' in df.columns:
            df['is_confirmation_bias'] = df['pre_search_confidence'].isin(['STRONG_HYPOTHESIS', 'WEAK_GUESS']) & df['is_search_query_biased']

    if 'is_utilization_failure' not in df.columns:
        if 'snippet_has_answer' in df.columns and 'agent_correct' in df.columns:
            df['is_utilization_failure'] = df['snippet_has_answer'] & (~df['agent_correct'])

    return df

analysis_df = load_and_preprocess(CSV_PATH, JSON_DIR)
analysis_df.columns

Loaded raw data: 97 rows.
Found 5 no-search run files.
Identified 11 questions with stable correct answers (5/5).
Updated 'baseline_correct' based on no-search JSON evaluations (5/5 correct).


Index(['problem_id', 'agent_correct', 'pre_search_confidence',
       'judge1_hypothesis', 'snippet_has_answer', 'answer_flipped',
       'is_search_query_biased', 'hypothesis_correct', 'baseline_correct',
       'is_context_poisoning', 'is_performative_ignorance',
       'is_confirmation_bias', 'is_utilization_failure'],
      dtype='str')

In [11]:
# Data already loaded in previous cell
analysis_df.columns

Index(['problem_id', 'agent_correct', 'pre_search_confidence',
       'judge1_hypothesis', 'snippet_has_answer', 'answer_flipped',
       'is_search_query_biased', 'hypothesis_correct', 'baseline_correct',
       'is_context_poisoning', 'is_performative_ignorance',
       'is_confirmation_bias', 'is_utilization_failure'],
      dtype='str')

In [12]:
traces = pd.read_json(TRACES_PATH)

def clean_problem(problem: str) -> str:
    separator = "\n\nYour response should be in the following format:"
    if separator in problem:
        return problem.split(separator)[0].strip()
    return problem.strip()

traces['problem_id'] = traces['problem'].apply(lambda problem: clean_problem(problem)[:50] + "...")
traces

,problem,agent_name,start_timestamp,end_timestamp,result,message_trace,metadata,problem_id
0,What was the original name of the insurance co...,baseline_agent,2026-02-05T16:16:41.184678Z,2026-02-05T16:16:58.408253Z,NaN,"[{'role': 'system', 'timestamp': None, 'finish...","{'total_messages': 5, 'tool_calls': 1, 'tool_r...",What was the original name of the insurance co...
1,In which Finnish town was a street named Valta...,baseline_agent,2026-02-05T16:16:11.038596Z,2026-02-05T16:16:38.691999Z,NaN,"[{'role': 'system', 'timestamp': None, 'finish...","{'total_messages': 11, 'tool_calls': 4, 'tool_...",In which Finnish town was a street named Valta...
2,Which television network broadcast the two par...,baseline_agent,2026-02-05T16:15:49.927509Z,2026-02-05T16:16:09.195759Z,NaN,"[{'role': 'system', 'timestamp': None, 'finish...","{'total_messages': 5, 'tool_calls': 1, 'tool_r...",Which television network broadcast the two par...
3,What is the name of the American textbook comp...,baseline_agent,2026-02-05T16:15:43.076897Z,2026-02-05T16:15:47.911812Z,NaN,"[{'role': 'system', 'timestamp': None, 'finish...","{'total_messages': 3, 'tool_calls': 0, 'tool_r...",What is the name of the American textbook comp...
4,What was the main religion practiced by the ma...,baseline_agent,2026-02-05T16:14:04.752752Z,2026-02-05T16:15:37.865498Z,NaN,"[{'role': 'system', 'timestamp': None, 'finish...","{'total_messages': 3, 'tool_calls': 0, 'tool_r...",What was the main religion practiced by the ma...
...,...,...,...,...,...,...,...,...
92,Which record label released the album titled '...,baseline_agent,2026-02-05T12:47:36.321809Z,2026-02-05T12:47:42.233130Z,NaN,"[{'role': 'system', 'timestamp': None, 'finish...","{'total_messages': 5, 'tool_calls': 1, 'tool_r...",Which record label released the album titled '...
93,What language were the Sinhala commentaries tr...,baseline_agent,2026-02-05T12:47:08.765301Z,2026-02-05T12:47:34.005933Z,NaN,"[{'role': 'system', 'timestamp': None, 'finish...","{'total_messages': 5, 'tool_calls': 1, 'tool_r...",What language were the Sinhala commentaries tr...
94,Which Hindu deity is also known as the lord of...,baseline_agent,2026-02-05T12:47:00.237171Z,2026-02-05T12:47:06.070434Z,NaN,"[{'role': 'system', 'timestamp': None, 'finish...","{'total_messages': 5, 'tool_calls': 1, 'tool_r...",Which Hindu deity is also known as the lord of...
95,In the election event which includes the '2024...,baseline_agent,2026-02-05T12:46:41.447096Z,2026-02-05T12:46:57.721850Z,NaN,"[{'role': 'system', 'timestamp': None, 'finish...","{'total_messages': 5, 'tool_calls': 1, 'tool_r...",In the election event which includes the '2024...


In [13]:
traces.iloc[0]['message_trace'][2]['parts'][0]['content']

'We need to solve the question: "What was the original name of the insurance company which is also known as Nationwide Mutual Group?" They want explanation, exact answer, confidence. We must use tools available: search with Brave Search API.\n\nThus we need to formulate a query that will yield the relevant info. Possibly searching for "Nationwide Mutual Group original name". Let\'s think: The insurance company referred to as Nationwide Mutual Group—maybe it\'s actually "The Nationwide Insurance Company"? Or originally known as "National Farmers Bank of New Albany"? Need recall. Actually, Nationwide is a large US insurance and financial services company, originally was called "The Farmers Mutual Insurance Company" maybe? Let’s search.\n\nBut rather than rely on my knowledge, we can use the search function to get results. The tool expects JSON with name "search" and parameters: query string, max_results optional integer up to 100 default? It says required: ["query"], so provide query; ma

In [14]:
for i, row in analysis_df[analysis_df['is_performative_ignorance']].iterrows():
    try:
        trace = traces[traces['problem_id'] == row['problem_id']]
        if trace.empty: print('Issue in row {}: No trace found for problem_id {}'.format(i, row['problem_id']))
        
        # agreement = analysis_df[analysis_df['problem_id'] == row['problem_id']]['baseline_correct'].iloc[0] if not analysis_df[analysis_df['problem_id'] == row['problem_id']].empty else 0

        # if not agreement:
        #     continue
        print("--- Question:", trace.iloc[0]['problem'])
        print(f"Agreement (Correct No-Search Runs): 5/5")
        print("Analysis:", row['pre_search_confidence'])
        print("Trace:")
        print(trace.iloc[0]['message_trace'][2]['parts'][0]['content'])
        print("\n" + "="*80 + "\n")
    except Exception as e:
        print(f"Error processing row {i}: {e}")

--- Question: According to a study performed at the educational institution also known as UNISI, Italy, what was the form of distraction used in the passive distraction experimental group during venipuncture?

Your response should be in the following format:
Explanation: {your explanation for your final answer}
Exact Answer: {your succinct, final answer}
Confidence: {your confidence score between 0% and 100% for your answer}
Agreement (Correct No-Search Runs): 5/5
Analysis: TABULA_RASA
Trace:
We need to answer a specific question about a study performed at the educational institution also known as UNISI, Italy. The study used venipuncture; they had a passive distraction experimental group; what was the form of distraction?

Thus we need to find information on this study via search. Likely it's about some research regarding distraction during venipuncture (blood draw) and they used different distraction methods - e.g., music, video, puzzle, etc. The "passive distraction" would probably 

In [15]:
for i, row in analysis_df[analysis_df['is_confirmation_bias']].iterrows():
    try:
        trace = traces[traces['problem_id'] == row['problem_id']]
        print("--- Question:", trace.iloc[0]['problem'] + "---")
        print("Analysis:", row['pre_search_confidence'])
        print("Trace:")
        print(trace.iloc[0]['message_trace'][2]['parts'][0]['content'])
        print(trace.iloc[0]['message_trace'][2]['parts'][1]['arguments']['query'])
        print("\n")
    except Exception as e:
        print(f"Error processing row {i}: {e}")

--- Question: In which state of the United States is the town of Danby, NY located?

Your response should be in the following format:
Explanation: {your explanation for your final answer}
Exact Answer: {your succinct, final answer}
Confidence: {your confidence score between 0% and 100% for your answer}---
Analysis: STRONG_HYPOTHESIS
Trace:
Okay, the user is asking which state in the US the town of Danby, NY is located in. Let me break this down.

First, I see that "Danby" is a town name followed by "NY". The "NY" here likely stands for New York. So the question is straightforward: Danby is a town in New York State. But to be thorough, maybe there's another place named Danby not in NY? Let me check.

I should use the search tool provided. The function requires a query and optionally max_results. The user wants the state of Danby, NY. So the query could be "Danby NY" or "Which state is Danby, NYC located?" Hmm, maybe better to just search for "Danby, New York". Wait, but sometimes cities

In [16]:
for i, row in analysis_df[analysis_df['is_utilization_failure']].iterrows():
    try:
        trace = traces[traces['problem_id'] == row['problem_id']]
        print("--- Question:", trace.iloc[0]['problem'] + "---")
        print("Analysis:", row['pre_search_confidence'])
        print("Trace:")
        print(trace.iloc[0]['message_trace'][2]['parts'][0]['content'])
        print("\n")
    except Exception as e:
        print(f"Error processing row {i}: {e}")

--- Question: What is the name of the road that enters Milabena, Tasmania, Australia from the north-east and runs through to the south before exiting?

Your response should be in the following format:
Explanation: {your explanation for your final answer}
Exact Answer: {your succinct, final answer}
Confidence: {your confidence score between 0% and 100% for your answer}---
Analysis: TABULA_RASA
Trace:
We need to answer a specific geography question. The user asks: "What is the name of the road that enters Milbena (maybe they wrote 'Milabena' but presumably 'Milford'? Actually it's "Milabena, Tasmania, Australia"? Let me parse.)

They are asking about a specific road that enters Milabena, Tasmania, Australia from north-east and runs through to the south before exiting. They want to know its name.

We need to find the name of the road via search, likely using a Search function. The only tool is "search". It performs Brave Search API automatically with pagination; can retrieve up to 100 res

In [17]:
for i, row in analysis_df[analysis_df['is_context_poisoning']].iterrows():
    try:
        trace = traces[traces['problem_id'] == row['problem_id']]
        print("--- Question:", trace.iloc[0]['problem'] + "---")
        print("Analysis:", row['pre_search_confidence'])
        print("Trace:")
        print(trace.iloc[0]['message_trace'][2]['parts'][0]['content'])
        print("\n")
    except Exception as e:
        print(f"Error processing row {i}: {e}")

--- Question: What term is used in modern Islamic finance for the basic financial instrument of the medieval Islamic world, where investors entrust capital to an agent for trade and profit sharing, with the agent not liable for losses not exceeding the subscribed capital?

Your response should be in the following format:
Explanation: {your explanation for your final answer}
Exact Answer: {your succinct, final answer}
Confidence: {your confidence score between 0% and 100% for your answer}---
Analysis: STRONG_HYPOTHESIS
Trace:
We need to answer a question about modern Islamic finance terminology related to medieval Islamic world's "basic financial instrument of the medieval Islamic world, where investors entrust capital to an agent for trade and profit sharing, with the agent not liable for losses not exceeding the subscribed capital". This sounds like "Mudaraba" or "Musharaka"? Let's think.

In classic Islamic finance, there are several profit-sharing contracts: Mudaraba (where a financ

### Exploring Unclassified No-Search Records
Identifying records where the agent did not perform a search and which do not fall into the established error categories (Context Poisoning, Performative Ignorance, Confirmation Bias, or Utilization Failure).

In [18]:
# Identify which records performed a search
traces['searched'] = traces['metadata'].apply(lambda x: x.get('tool_calls', 0) > 0)
analysis_with_search = analysis_df.merge(traces[['problem_id', 'searched']], on='problem_id', how='left')

# Define classified flags
classified_cols = ['is_context_poisoning', 'is_performative_ignorance', 'is_confirmation_bias', 'is_utilization_failure']
analysis_with_search['is_classified'] = analysis_with_search[classified_cols].any(axis=1)

# Filter for no-search records that are not classified
# We are particularly interested in failures (agent_correct == False)
unclassified_no_search_failures = analysis_with_search[
    (analysis_with_search['searched'] == False) & 
    (analysis_with_search['is_classified'] == False) & 
    (analysis_with_search['agent_correct'] == False)
]

print(f"Found {len(unclassified_no_search_failures)} unclassified no-search failures.")

for i, row in unclassified_no_search_failures.iterrows():
    try:
        trace = traces[traces['problem_id'] == row['problem_id']]
        print("--- Question:", trace.iloc[0]['problem'] + "---")
        print(f"Baseline Correct: {row['baseline_correct']}, Agent Correct: {row['agent_correct']}")
        print("Pre-search Confidence:", row['pre_search_confidence'])
        print("Trace (First Message):")
        # For no-search, let's see why it decided not to search
        print(trace.iloc[0]['message_trace'][2]['parts'][0]['content'][:1000] + "...")
        print("\n" + "="*80 + "\n")
    except Exception as e:
        print(f"Error processing row {i}: {e}")

Found 8 unclassified no-search failures.
--- Question: According to the Venetic theory, which people are considered the last remnant of the original European tribe who spoke Gaulish and dwelled in Armorica, in the southern part of the Brittany Peninsula, during the Iron Age and the Roman period?

Your response should be in the following format:
Explanation: {your explanation for your final answer}
Exact Answer: {your succinct, final answer}
Confidence: {your confidence score between 0% and 100% for your answer}---
Baseline Correct: False, Agent Correct: False
Pre-search Confidence: nan
Trace (First Message):
The user asks:

"According to the Venetic theory, which people are considered the last remnant of the original European tribe who spoke Gaulish and dwelled in Armorica, in the southern part of the Brittany Peninsula, during the Iron Age and the Roman period?"

They want a succinct answer formatted exactly as:
Explanation: ...
Exact Answer: ...
Confidence: ...

We need to interpret 